# Create MMVEC tbls 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [77]:
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [78]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [79]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [80]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [81]:
# skin_group = "Acne_L"
# skin_group = "Acne_NL"
skin_group = "Healthy"

In [82]:
# Read in metabolomics table and set index
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/data_sample_10282024.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in VIP-filtered feature list
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_with_shortened_names.tsv', sep='\t')

# Get list of VIP feature IDs
vip_features = VIP_filtered['ID'].astype(str).unique().tolist()

# Extract the feature ID from each column name (assumes feature ID is before the first semicolon)
col_ids = [col.split(';')[0] for col in metaB_tbl.columns]

# Map original column names to their extracted feature IDs
col_map = dict(zip(metaB_tbl.columns, col_ids))

# Keep only columns where the extracted ID matches a VIP feature
cols_to_keep = [col for col, cid in col_map.items() if cid in vip_features]

# Subset the table to keep only those columns
metaB_tbl = metaB_tbl[cols_to_keep]

# Create mapping of ID to Shortened_Compound_Name
VIP_filtered['ID'] = VIP_filtered['ID'].astype(str)  # Ensure IDs are strings
name_map = VIP_filtered.set_index('ID')['Shortened_Compound_Name'].to_dict()

# Rename columns to Shortened_Compound_Name only (no mz, no RT)
new_names = [name_map.get(col.split(';')[0], col.split(';')[0]) for col in metaB_tbl.columns]
metaB_tbl.columns = new_names

# Identify and print duplicates (i.e., metabolites to collapse)
from collections import Counter

name_counts = Counter(new_names)
collapsed = {name: count for name, count in name_counts.items() if count > 1}

if collapsed:
    print("The following metabolites had multiple columns and were collapsed by averaging:")
    for name, count in collapsed.items():
        print(f"- {name}: {count} columns")
else:
    print("No metabolites were duplicated and collapsed.")

# Collapse duplicate columns by averaging
metaB_tbl = metaB_tbl.groupby(metaB_tbl.columns, axis=1).mean()

# View the result
metaB_tbl


The following metabolites had multiple columns and were collapsed by averaging:
- (Iso)leucine: 2 columns
- Urocanic acid: 2 columns


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20648/3082422782.py:45: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  metaB_tbl = metaB_tbl.groupby(metaB_tbl.columns, axis=1).mean()


,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,1.043807e+06,0.0000,611709.700,302670.840,1473068.10,0.0000,12406.8260,16778.838,125692.3300,0.00,...,970487.500,0.000,2987383.80,0.000,0.00,567510.060,1942604.500,50026.523,34717.20300,0.0
LAMI.RD306.D9.C2,1.206857e+06,0.0000,437938.200,472807.600,308019.97,0.0000,20433.4900,12865.390,10553.9270,178911.47,...,1071868.200,0.000,2051791.50,0.000,0.00,672054.400,3572847.500,74657.664,17233.24200,0.0
LAMI.RD308.D2.C2,8.785843e+05,3822.6377,231525.840,179426.700,286004.00,0.0000,5348.9400,14299.352,4734.5693,0.00,...,750870.100,36691.316,1121916.40,0.000,0.00,442653.660,1618142.800,39865.797,2985.37550,0.0
LAMI.RD304.D11.C1,7.463111e+05,29824.6270,78491.530,218959.970,299140.06,5375.8525,9403.6520,0.000,12473.0920,0.00,...,595568.940,0.000,1818308.60,11225.194,99470.69,328151.120,1605551.000,26177.530,42064.83650,0.0
LAMI.RD304.D11.C2,3.631755e+05,44084.5860,261996.800,119362.910,191713.67,2691.0256,2286.2397,15780.309,40673.1100,0.00,...,303845.300,0.000,1126503.20,16378.752,157503.92,152392.720,865753.500,19284.926,1730.04870,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,6.373643e+05,0.0000,122958.260,188968.060,236111.75,0.0000,5144.6616,56654.043,0.0000,0.00,...,579058.800,0.000,1173415.00,0.000,0.00,498893.780,1380958.500,27379.004,9200.49000,0.0
LAMI.RD308.D9.C3,3.560037e+05,0.0000,65659.700,80050.120,303509.40,0.0000,0.0000,21826.117,220846.1600,0.00,...,281896.700,658741.100,550094.94,0.000,0.00,262886.470,742416.600,21597.020,3120.44365,0.0
LAMI.RD319.D2.C2,2.868188e+05,0.0000,73231.560,62017.848,5406879.00,0.0000,0.0000,19741.246,25695.0140,0.00,...,255542.720,9720.738,602147.10,0.000,0.00,121803.890,1194061.900,16367.659,2003.32420,0.0


In [83]:
# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)
# Extract the feature name after the last semicolon for each column
# V1V3_tbl.columns = V1V3_tbl.columns.str.split(';').str[-1].str.strip()
# # Sort columns by their sum in descending order
# V1V3_tbl = V1V3_tbl.loc[:, V1V3_tbl.sum().sort_values(ascending=False).index]


# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [84]:
V1V3_tbl

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,...,g__Ramlibacter,g__Delftia,g__Lacibacter,g__Luteitalea,g__Pseudochrobactrum,g__Methylotenera,g__Fusicatenibacter,g__Ruminococcus,g__Lachnospira,g__Parabacteroides
LAMI.RD001.D0.C1,2362.0,0.0,94.0,292.0,175.0,91.0,33.0,37.0,18.0,1.0,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D0.C2,1808.0,2.0,158.0,374.0,352.0,120.0,36.0,52.0,27.0,165.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C1,2234.0,2.0,83.0,240.0,492.0,253.0,19.0,30.0,30.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C2,1761.0,0.0,137.0,446.0,456.0,151.0,69.0,16.0,100.0,26.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D28.C1,2367.0,11.0,118.0,293.0,365.0,217.0,34.0,22.0,38.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D14.C1,1900.0,1846.0,8.0,0.0,6.0,8.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C3,1003.0,2741.0,11.0,1.0,4.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,1410.0,8.0,0.0,9.0,27.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D9.C1,1230.0,2491.0,13.0,1.0,3.0,31.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [85]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
# metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
LAMI.RD001.D0.C1,1.043807e+06,0.000,611709.700,302670.840,1473068.10,0.0000,12406.8260,16778.8380,125692.330,0.00,...,970487.500,0.000,2987383.80,0.000,0.00,567510.060,1942604.500,50026.523,34717.20300,0.0
LAMI.RD306.D9.C2,1.206857e+06,0.000,437938.200,472807.600,308019.97,0.0000,20433.4900,12865.3900,10553.927,178911.47,...,1071868.200,0.000,2051791.50,0.000,0.00,672054.400,3572847.500,74657.664,17233.24200,0.0
LAMI.RD304.D11.C1,7.463111e+05,29824.627,78491.530,218959.970,299140.06,5375.8525,9403.6520,0.0000,12473.092,0.00,...,595568.940,0.000,1818308.60,11225.194,99470.69,328151.120,1605551.000,26177.530,42064.83650,0.0
LAMI.RD304.D11.C2,3.631755e+05,44084.586,261996.800,119362.910,191713.67,2691.0256,2286.2397,15780.3090,40673.110,0.00,...,303845.300,0.000,1126503.20,16378.752,157503.92,152392.720,865753.500,19284.926,1730.04870,0.0
LAMI.RD304.D0.C1,4.118800e+05,0.000,427202.060,132853.970,705570.80,3431.0034,0.0000,3228.1143,49464.902,0.00,...,293386.340,0.000,827466.30,0.000,0.00,137201.390,1321474.400,15696.905,3946.54120,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,6.373643e+05,0.000,122958.260,188968.060,236111.75,0.0000,5144.6616,56654.0430,0.000,0.00,...,579058.800,0.000,1173415.00,0.000,0.00,498893.780,1380958.500,27379.004,9200.49000,0.0
LAMI.RD308.D9.C3,3.560037e+05,0.000,65659.700,80050.120,303509.40,0.0000,0.0000,21826.1170,220846.160,0.00,...,281896.700,658741.100,550094.94,0.000,0.00,262886.470,742416.600,21597.020,3120.44365,0.0
LAMI.RD319.D2.C2,2.868188e+05,0.000,73231.560,62017.848,5406879.00,0.0000,0.0000,19741.2460,25695.014,0.00,...,255542.720,9720.738,602147.10,0.000,0.00,121803.890,1194061.900,16367.659,2003.32420,0.0
LAMI.RD319.D2.C3,6.612208e+04,0.000,55986.290,32995.594,2838344.20,0.0000,0.0000,0.0000,11039.730,0.00,...,59064.336,0.000,171846.72,0.000,0.00,23485.404,833009.000,12853.001,703.76150,0.0


In [86]:
# Convert to relative abundance (row-wise normalization)
metaB_V1V3_tbl_relative = metaB_V1V3_tbl.div(metaB_V1V3_tbl.sum(axis=1), axis=0)
metaB_V1V3_tbl_relative

,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
LAMI.RD001.D0.C1,0.102665,0.000000,0.060165,0.029770,0.144885,0.000000,0.001220,0.001650,0.012363,0.000000,...,0.095453,0.000000,0.293827,0.000000,0.000000,0.055818,0.191067,0.004920,0.003415,0.0
LAMI.RD306.D9.C2,0.116470,0.000000,0.042264,0.045629,0.029726,0.000000,0.001972,0.001242,0.001019,0.017266,...,0.103443,0.000000,0.198013,0.000000,0.000000,0.064858,0.344805,0.007205,0.001663,0.0
LAMI.RD304.D11.C1,0.125106,0.005000,0.013158,0.036705,0.050146,0.000901,0.001576,0.000000,0.002091,0.000000,...,0.099837,0.000000,0.304807,0.001882,0.016675,0.055009,0.269142,0.004388,0.007051,0.0
LAMI.RD304.D11.C2,0.097235,0.011803,0.070146,0.031958,0.051328,0.000720,0.000612,0.004225,0.010890,0.000000,...,0.081350,0.000000,0.301604,0.004385,0.042169,0.040801,0.231793,0.005163,0.000463,0.0
LAMI.RD304.D0.C1,0.094080,0.000000,0.097580,0.030346,0.161163,0.000784,0.000000,0.000737,0.011299,0.000000,...,0.067014,0.000000,0.189006,0.000000,0.000000,0.031339,0.301845,0.003585,0.000901,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.129261,0.000000,0.024937,0.038324,0.047885,0.000000,0.001043,0.011490,0.000000,0.000000,...,0.117437,0.000000,0.237976,0.000000,0.000000,0.101179,0.280067,0.005553,0.001866,0.0
LAMI.RD308.D9.C3,0.099260,0.000000,0.018307,0.022319,0.084624,0.000000,0.000000,0.006085,0.061576,0.000000,...,0.078598,0.183668,0.153376,0.000000,0.000000,0.073297,0.206998,0.006022,0.000870,0.0
LAMI.RD319.D2.C2,0.035468,0.000000,0.009056,0.007669,0.668623,0.000000,0.000000,0.002441,0.003177,0.000000,...,0.031601,0.001202,0.074462,0.000000,0.000000,0.015062,0.147660,0.002024,0.000248,0.0
LAMI.RD319.D2.C3,0.016007,0.000000,0.013553,0.007988,0.687114,0.000000,0.000000,0.000000,0.002673,0.000000,...,0.014298,0.000000,0.041601,0.000000,0.000000,0.005685,0.201657,0.003111,0.000170,0.0


In [87]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V1V3_tbl_relative.index = metaB_V1V3_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V1V3_tbl_relative.index)]

# Subset the DataFrame
metaB_V1V3_tbl_relative = metaB_V1V3_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V1V3_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20648/3342871550.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
LAMI.RD002.D14.C1,0.096730,0.021292,0.065954,0.017741,0.076557,0.0,0.000435,0.001193,0.000000,0.000000,...,0.089154,0.000000,0.194910,0.000000,0.000000,0.037673,0.367050,0.009656,0.001578,0.000000
LAMI.RD003.D14.C1,0.021133,0.000000,0.001543,0.007920,0.807134,0.0,0.000000,0.000292,0.001731,0.000000,...,0.015214,0.000000,0.044914,0.000000,0.000000,0.007074,0.087052,0.001175,0.000000,0.000000
LAMI.RD017.D0.C2,0.113992,0.000000,0.042071,0.042788,0.037336,0.0,0.000000,0.004434,0.000648,0.002995,...,0.107703,0.000000,0.251728,0.000000,0.004384,0.062822,0.320093,0.006240,0.000555,0.000000
LAMI.RD007.D14.C1,0.098458,0.000000,0.060199,0.013305,0.105061,0.0,0.001091,0.003353,0.019092,0.011904,...,0.083775,0.000000,0.227457,0.000000,0.000000,0.030555,0.332267,0.006231,0.000745,0.000000
LAMI.RD017.D28.C2,0.151693,0.000000,0.028205,0.024962,0.013945,0.0,0.002528,0.004435,0.000386,0.012115,...,0.147447,0.000000,0.278067,0.000000,0.001740,0.089243,0.236231,0.007561,0.000311,0.000000
LAMI.RD002.D28.C2,0.013363,0.000000,0.000181,0.001552,0.820976,0.0,0.000034,0.001072,0.004738,0.001385,...,0.010325,0.000000,0.046343,0.000000,0.001203,0.006406,0.088115,0.001340,0.000035,0.000000
LAMI.RD004.D0.C1,0.054631,0.000000,0.062046,0.022727,0.276661,0.0,0.000178,0.000000,0.000000,0.020702,...,0.040781,0.000000,0.187259,0.000000,0.004563,0.030067,0.292748,0.003105,0.001088,0.000000
LAMI.RD003.D28.C1,0.061483,0.000000,0.004013,0.043865,0.403936,0.0,0.000380,0.004155,0.000000,0.000000,...,0.046627,0.000000,0.223188,0.000000,0.000000,0.031494,0.175373,0.001155,0.000399,0.000000
LAMI.RD001.D14.C1,0.136200,0.000000,0.091800,0.052434,0.040587,0.0,0.003525,0.000409,0.000000,0.001012,...,0.120480,0.001390,0.263190,0.000000,0.000000,0.056847,0.222364,0.004741,0.003330,0.000000
LAMI.RD001.D28.C1,0.114040,0.000000,0.017378,0.053930,0.024180,0.0,0.005706,0.001405,0.002728,0.000301,...,0.116397,0.000193,0.330000,0.000000,0.002334,0.083988,0.233306,0.007324,0.002927,0.000000


In [88]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V1V3_tbl_relative, output_path)

In [89]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl_relative.index)]

top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
                     ' g__Streptococcus',' g__Lawsonella',
                     ' g__Micrococcus', ' g__Lactobacillus', 
                     ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']

# top_V1V3_features = ['g__Cutibacterium|ASV1', 'g__Staphylococcus|ASV1',
#                      'g__Streptococcus|ASV1','g__Lawsonella',
#                      ' g__Micrococcus|ASV1', 'g__Lactobacillus|ASV1', 
#                      'g__Rothia|ASV1', 'g__Kocuria|ASV1', 'g__Haemophilus|ASV1', 'g__Corynebacterium|ASV1']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl_matched.columns.intersection(top_V1V3_features)
V1V3_tbl_matched = V1V3_tbl_matched[available_columns]

V1V3_tbl_matched

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,2362.0,94.0,292.0,175.0,91.0,37.0,1.0,50.0,87.0,0.0
LAMI.RD001.D0.C2,1808.0,158.0,374.0,352.0,120.0,52.0,165.0,43.0,74.0,9.0
LAMI.RD001.D14.C1,2234.0,83.0,240.0,492.0,253.0,30.0,60.0,12.0,8.0,0.0
LAMI.RD001.D14.C2,1761.0,137.0,446.0,456.0,151.0,16.0,26.0,31.0,58.0,6.0
LAMI.RD001.D28.C1,2367.0,118.0,293.0,365.0,217.0,22.0,17.0,14.0,10.0,5.0
LAMI.RD002.D0.C2,2900.0,373.0,159.0,39.0,13.0,61.0,1.0,27.0,11.0,54.0
LAMI.RD003.D14.C1,3296.0,95.0,14.0,138.0,14.0,0.0,1.0,5.0,0.0,0.0
LAMI.RD002.D14.C1,3440.0,105.0,43.0,49.0,41.0,7.0,2.0,4.0,1.0,0.0
LAMI.RD003.D28.C1,2367.0,103.0,543.0,31.0,36.0,0.0,0.0,16.0,21.0,0.0
LAMI.RD001.D28.C2,2288.0,204.0,169.0,399.0,420.0,20.0,11.0,20.0,11.0,9.0


In [90]:
# Convert to relative abundance (row-wise normalization)
V1V3_tbl_matched_relative = V1V3_tbl_matched.div(V1V3_tbl_matched.sum(axis=1), axis=0)
V1V3_tbl_matched_relative

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,0.740671,0.029476,0.091565,0.054876,0.028536,0.011602,0.000314,0.015679,0.027281,0.000000
LAMI.RD001.D0.C2,0.573059,0.050079,0.118542,0.111569,0.038035,0.016482,0.052298,0.013629,0.023455,0.002853
LAMI.RD001.D14.C1,0.654748,0.024326,0.070340,0.144197,0.074150,0.008792,0.017585,0.003517,0.002345,0.000000
LAMI.RD001.D14.C2,0.570272,0.044365,0.144430,0.147668,0.048899,0.005181,0.008420,0.010039,0.018782,0.001943
LAMI.RD001.D28.C1,0.690490,0.034422,0.085473,0.106476,0.063302,0.006418,0.004959,0.004084,0.002917,0.001459
LAMI.RD002.D0.C2,0.797141,0.102529,0.043705,0.010720,0.003573,0.016767,0.000275,0.007422,0.003024,0.014843
LAMI.RD003.D14.C1,0.925063,0.026663,0.003929,0.038731,0.003929,0.000000,0.000281,0.001403,0.000000,0.000000
LAMI.RD002.D14.C1,0.931744,0.028440,0.011647,0.013272,0.011105,0.001896,0.000542,0.001083,0.000271,0.000000
LAMI.RD003.D28.C1,0.759384,0.033045,0.174206,0.009945,0.011550,0.000000,0.000000,0.005133,0.006737,0.000000
LAMI.RD001.D28.C2,0.644326,0.057449,0.047592,0.112363,0.118277,0.005632,0.003098,0.005632,0.003098,0.002534


In [91]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched_relative.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
9,LAMI.RD002.D14.C1,Healthy
14,LAMI.RD003.D14.C1,Healthy
22,LAMI.RD017.D0.C2,Healthy
27,LAMI.RD007.D14.C1,Healthy
28,LAMI.RD017.D28.C2,Healthy
30,LAMI.RD002.D28.C2,Healthy
33,LAMI.RD004.D0.C1,Healthy
36,LAMI.RD003.D28.C1,Healthy
38,LAMI.RD001.D14.C1,Healthy
45,LAMI.RD001.D28.C1,Healthy


In [92]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched_relative.T
output_path = f'../Data/multi-omics/16S_V1V3-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [93]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,(Iso)leucine
1,3-O-methylated flavonoids
2,C19H36O4 Fatty alcohol
3,Cyclo(his-pro)
4,DL-Panthenol
5,Gln-C14:0
6,Glutamyltyrosine
7,N-Oleoylethanolamine
8,NCGC00380271-01
9,Nicotine


In [94]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [95]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
# metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
LAMI.RD001.D0.C1,1.043807e+06,0.000,611709.700,302670.840,1473068.10,0.0000,12406.8260,16778.8380,125692.330,0.00,...,970487.500,0.000,2987383.80,0.000,0.00,567510.060,1942604.500,50026.523,34717.20300,0.0
LAMI.RD306.D9.C2,1.206857e+06,0.000,437938.200,472807.600,308019.97,0.0000,20433.4900,12865.3900,10553.927,178911.47,...,1071868.200,0.000,2051791.50,0.000,0.00,672054.400,3572847.500,74657.664,17233.24200,0.0
LAMI.RD304.D11.C1,7.463111e+05,29824.627,78491.530,218959.970,299140.06,5375.8525,9403.6520,0.0000,12473.092,0.00,...,595568.940,0.000,1818308.60,11225.194,99470.69,328151.120,1605551.000,26177.530,42064.83650,0.0
LAMI.RD304.D11.C2,3.631755e+05,44084.586,261996.800,119362.910,191713.67,2691.0256,2286.2397,15780.3090,40673.110,0.00,...,303845.300,0.000,1126503.20,16378.752,157503.92,152392.720,865753.500,19284.926,1730.04870,0.0
LAMI.RD304.D0.C1,4.118800e+05,0.000,427202.060,132853.970,705570.80,3431.0034,0.0000,3228.1143,49464.902,0.00,...,293386.340,0.000,827466.30,0.000,0.00,137201.390,1321474.400,15696.905,3946.54120,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,6.373643e+05,0.000,122958.260,188968.060,236111.75,0.0000,5144.6616,56654.0430,0.000,0.00,...,579058.800,0.000,1173415.00,0.000,0.00,498893.780,1380958.500,27379.004,9200.49000,0.0
LAMI.RD308.D9.C3,3.560037e+05,0.000,65659.700,80050.120,303509.40,0.0000,0.0000,21826.1170,220846.160,0.00,...,281896.700,658741.100,550094.94,0.000,0.00,262886.470,742416.600,21597.020,3120.44365,0.0
LAMI.RD319.D2.C2,2.868188e+05,0.000,73231.560,62017.848,5406879.00,0.0000,0.0000,19741.2460,25695.014,0.00,...,255542.720,9720.738,602147.10,0.000,0.00,121803.890,1194061.900,16367.659,2003.32420,0.0
LAMI.RD319.D2.C3,6.612208e+04,0.000,55986.290,32995.594,2838344.20,0.0000,0.0000,0.0000,11039.730,0.00,...,59064.336,0.000,171846.72,0.000,0.00,23485.404,833009.000,12853.001,703.76150,0.0


In [96]:
# Drop the column from the DataFrame as it wasn't significant after univariate testing
# metaB_V4_tbl = metaB_V4_tbl.drop(columns=['C19H22O4 Linear diarylheptanoids'])

In [97]:
# Convert to relative abundance (row-wise normalization)
metaB_V4_tbl_relative = metaB_V4_tbl.div(metaB_V4_tbl.sum(axis=1), axis=0)
metaB_V4_tbl_relative

,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
LAMI.RD001.D0.C1,0.102665,0.000000,0.060165,0.029770,0.144885,0.000000,0.001220,0.001650,0.012363,0.000000,...,0.095453,0.000000,0.293827,0.000000,0.000000,0.055818,0.191067,0.004920,0.003415,0.0
LAMI.RD306.D9.C2,0.116470,0.000000,0.042264,0.045629,0.029726,0.000000,0.001972,0.001242,0.001019,0.017266,...,0.103443,0.000000,0.198013,0.000000,0.000000,0.064858,0.344805,0.007205,0.001663,0.0
LAMI.RD304.D11.C1,0.125106,0.005000,0.013158,0.036705,0.050146,0.000901,0.001576,0.000000,0.002091,0.000000,...,0.099837,0.000000,0.304807,0.001882,0.016675,0.055009,0.269142,0.004388,0.007051,0.0
LAMI.RD304.D11.C2,0.097235,0.011803,0.070146,0.031958,0.051328,0.000720,0.000612,0.004225,0.010890,0.000000,...,0.081350,0.000000,0.301604,0.004385,0.042169,0.040801,0.231793,0.005163,0.000463,0.0
LAMI.RD304.D0.C1,0.094080,0.000000,0.097580,0.030346,0.161163,0.000784,0.000000,0.000737,0.011299,0.000000,...,0.067014,0.000000,0.189006,0.000000,0.000000,0.031339,0.301845,0.003585,0.000901,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.129261,0.000000,0.024937,0.038324,0.047885,0.000000,0.001043,0.011490,0.000000,0.000000,...,0.117437,0.000000,0.237976,0.000000,0.000000,0.101179,0.280067,0.005553,0.001866,0.0
LAMI.RD308.D9.C3,0.099260,0.000000,0.018307,0.022319,0.084624,0.000000,0.000000,0.006085,0.061576,0.000000,...,0.078598,0.183668,0.153376,0.000000,0.000000,0.073297,0.206998,0.006022,0.000870,0.0
LAMI.RD319.D2.C2,0.035468,0.000000,0.009056,0.007669,0.668623,0.000000,0.000000,0.002441,0.003177,0.000000,...,0.031601,0.001202,0.074462,0.000000,0.000000,0.015062,0.147660,0.002024,0.000248,0.0
LAMI.RD319.D2.C3,0.016007,0.000000,0.013553,0.007988,0.687114,0.000000,0.000000,0.000000,0.002673,0.000000,...,0.014298,0.000000,0.041601,0.000000,0.000000,0.005685,0.201657,0.003111,0.000170,0.0


In [98]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V4_tbl_relative.index = metaB_V4_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V4_tbl_relative.index)]

# Subset the DataFrame
metaB_V4_tbl_relative = metaB_V4_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V4_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20648/4148729606.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,(Iso)leucine,3-O-methylated flavonoids,C19H36O4 Fatty alcohol,Cyclo(his-pro),DL-Panthenol,Gln-C14:0,Glutamyltyrosine,N-Oleoylethanolamine,NCGC00380271-01,Nicotine,...,Phenylalanine,Phenylbenzimidazole sulfonic acid,Pyroglutamic acid,Sinensetin,Sorbitane Monooleate,Tryptophan,Tyrosine,Uridine,Urocanic acid,xylometazoline
LAMI.RD017.D0.C1,0.114963,0.0,0.034712,0.041308,0.049563,0.000497,0.000000,0.003912,0.000000,0.001300,...,0.117187,0.000000,0.298382,0.0,0.004950,0.077045,0.249181,0.004129,0.000446,0.000000
LAMI.RD017.D0.C2,0.113992,0.0,0.042071,0.042788,0.037336,0.000000,0.000000,0.004434,0.000648,0.002995,...,0.107703,0.000000,0.251728,0.0,0.004384,0.062822,0.320093,0.006240,0.000555,0.000000
LAMI.RD007.D14.C1,0.098458,0.0,0.060199,0.013305,0.105061,0.000000,0.001091,0.003353,0.019092,0.011904,...,0.083775,0.000000,0.227457,0.0,0.000000,0.030555,0.332267,0.006231,0.000745,0.000000
LAMI.RD017.D28.C2,0.151693,0.0,0.028205,0.024962,0.013945,0.000000,0.002528,0.004435,0.000386,0.012115,...,0.147447,0.000000,0.278067,0.0,0.001740,0.089243,0.236231,0.007561,0.000311,0.000000
LAMI.RD004.D0.C1,0.054631,0.0,0.062046,0.022727,0.276661,0.000000,0.000178,0.000000,0.000000,0.020702,...,0.040781,0.000000,0.187259,0.0,0.004563,0.030067,0.292748,0.003105,0.001088,0.000000
LAMI.RD001.D14.C1,0.136200,0.0,0.091800,0.052434,0.040587,0.000000,0.003525,0.000409,0.000000,0.001012,...,0.120480,0.001390,0.263190,0.0,0.000000,0.056847,0.222364,0.004741,0.003330,0.000000
LAMI.RD001.D28.C1,0.114040,0.0,0.017378,0.053930,0.024180,0.000000,0.005706,0.001405,0.002728,0.000301,...,0.116397,0.000193,0.330000,0.0,0.002334,0.083988,0.233306,0.007324,0.002927,0.000000
LAMI.RD017.D28.C1,0.151028,0.0,0.030628,0.022384,0.025823,0.000000,0.001889,0.003842,0.000000,0.012900,...,0.135794,0.000227,0.254026,0.0,0.002122,0.085862,0.264601,0.005059,0.001699,0.000000
LAMI.RD017.D14.C2,0.129811,0.0,0.030035,0.020961,0.109908,0.000310,0.001554,0.001684,0.002989,0.002891,...,0.110605,0.000000,0.301604,0.0,0.000825,0.063217,0.214513,0.006515,0.000927,0.000000
LAMI.RD006.D14.C1,0.020101,0.0,0.000320,0.003610,0.794958,0.000000,0.000000,0.001975,0.000552,0.009064,...,0.014816,0.001612,0.088845,0.0,0.000000,0.008629,0.053753,0.001153,0.000175,0.000000


In [99]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_V4_tbl_relative.index)]

# top_V4_features = [' g__Staphylococcus', ' g__Lawsonella',
#                     ' g__Streptococcus', ' g__Acinetobacter', ' g__Cutibacterium', 'g__Enhydrobacter',
#                      ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']


top_V4_features = [' g__Staphylococcus', ' g__Lawsonella', ' g__Streptococcus', ' g__Cutibacterium',
                     ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V4_tbl_matched.columns.intersection(top_V4_features)
V4_tbl_matched = V4_tbl_matched[available_columns]

V4_tbl_matched

,g__Staphylococcus,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,317.0,131.0,275.0,414.0,475.0,100.0,15.0,1.0,17.0,36.0
LAMI.RD001.D14.C1,454.0,456.0,915.0,247.0,168.0,93.0,117.0,0.0,5.0,15.0
LAMI.RD004.D0.C2,1652.0,95.0,848.0,164.0,28.0,81.0,97.0,1.0,18.0,49.0
LAMI.RD001.D0.C2,349.0,217.0,509.0,368.0,464.0,75.0,182.0,3.0,3.0,56.0
LAMI.RD011.D0.C2,2132.0,291.0,220.0,153.0,227.0,60.0,26.0,3.0,44.0,27.0
LAMI.RD001.D14.C2,444.0,206.0,546.0,457.0,456.0,54.0,23.0,0.0,5.0,47.0
LAMI.RD004.D28.C2,217.0,18.0,731.0,713.0,44.0,276.0,40.0,2.0,16.0,15.0
LAMI.RD017.D14.C2,1756.0,636.0,131.0,33.0,25.0,617.0,2.0,126.0,7.0,1.0
LAMI.RD011.D14.C1,2063.0,999.0,102.0,85.0,37.0,88.0,45.0,32.0,3.0,7.0
LAMI.RD004.D0.C1,603.0,48.0,875.0,270.0,14.0,83.0,21.0,6.0,5.0,30.0


In [100]:
# Convert to relative abundance (row-wise normalization)
V4_tbl_matched_relative = V4_tbl_matched.div(V4_tbl_matched.sum(axis=1), axis=0)
V4_tbl_matched_relative

,g__Staphylococcus,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,0.177990,0.073554,0.154408,0.232454,0.266704,0.056148,0.008422,0.000561,0.009545,0.020213
LAMI.RD001.D14.C1,0.183806,0.184615,0.370445,0.100000,0.068016,0.037652,0.047368,0.000000,0.002024,0.006073
LAMI.RD004.D0.C2,0.544675,0.031322,0.279591,0.054072,0.009232,0.026706,0.031982,0.000330,0.005935,0.016156
LAMI.RD001.D0.C2,0.156783,0.097484,0.228661,0.165319,0.208446,0.033693,0.081761,0.001348,0.001348,0.025157
LAMI.RD011.D0.C2,0.669808,0.091423,0.069117,0.048068,0.071316,0.018850,0.008168,0.000943,0.013823,0.008483
LAMI.RD001.D14.C2,0.198391,0.092046,0.243968,0.204200,0.203753,0.024129,0.010277,0.000000,0.002234,0.021001
LAMI.RD004.D28.C2,0.104730,0.008687,0.352799,0.344112,0.021236,0.133205,0.019305,0.000965,0.007722,0.007239
LAMI.RD017.D14.C2,0.526695,0.190762,0.039292,0.009898,0.007499,0.185063,0.000600,0.037792,0.002100,0.000300
LAMI.RD011.D14.C1,0.596070,0.288645,0.029471,0.024559,0.010691,0.025426,0.013002,0.009246,0.000867,0.002023
LAMI.RD004.D0.C1,0.308440,0.024552,0.447570,0.138107,0.007161,0.042455,0.010742,0.003069,0.002558,0.015345


In [101]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V4.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V4.columns[1:])]

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
18,LAMI.RD017.D0.C1,Healthy
22,LAMI.RD017.D0.C2,Healthy
27,LAMI.RD007.D14.C1,Healthy
28,LAMI.RD017.D28.C2,Healthy
33,LAMI.RD004.D0.C1,Healthy
38,LAMI.RD001.D14.C1,Healthy
45,LAMI.RD001.D28.C1,Healthy
46,LAMI.RD017.D28.C1,Healthy
49,LAMI.RD017.D14.C2,Healthy
58,LAMI.RD006.D14.C1,Healthy


In [102]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V4-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [103]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched_relative.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

### Combine V1-V3 and V4 tables

In [104]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Merge by taking average of values row-wise
merged_tbl = V1V3_tbl_matched.add(V4_tbl_matched, fill_value=0)

# Sort columns by total sum (descending order)
merged_tbl = merged_tbl[merged_tbl.sum().sort_values(ascending=False).index]

merged_tbl

,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,2363.0,411.0,450.0,706.0,222.0,137.0,562.0,17.0,16.0,86.0
LAMI.RD001.D0.C2,1811.0,507.0,861.0,742.0,337.0,127.0,538.0,12.0,347.0,99.0
LAMI.RD001.D14.C1,2234.0,537.0,1407.0,487.0,709.0,123.0,176.0,5.0,177.0,27.0
LAMI.RD001.D14.C2,1761.0,581.0,1002.0,903.0,357.0,70.0,514.0,11.0,49.0,78.0
LAMI.RD001.D28.C1,2368.0,698.0,1057.0,642.0,674.0,94.0,152.0,10.0,100.0,48.0
LAMI.RD001.D28.C2,2292.0,1053.0,1261.0,448.0,913.0,97.0,125.0,14.0,41.0,45.0
LAMI.RD007.D14.C1,3647.0,2094.0,297.0,183.0,367.0,40.0,78.0,0.0,7.0,31.0
LAMI.RD007.D14.C2,3598.0,1567.0,379.0,168.0,507.0,63.0,141.0,0.0,0.0,0.0
LAMI.RD004.D0.C1,2680.0,705.0,1329.0,580.0,70.0,104.0,17.0,5.0,32.0,59.0
LAMI.RD006.D14.C2,3400.0,774.0,917.0,131.0,416.0,156.0,15.0,33.0,132.0,3.0


In [105]:
V1V3_tbl_matched.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Streptococcus',
       ' g__Corynebacterium', ' g__Lawsonella', ' g__Micrococcus',
       ' g__Lactobacillus', ' g__Rothia', ' g__Haemophilus', ' g__Kocuria'],
      dtype='object')

In [106]:
V4_tbl_matched.columns

Index([' g__Staphylococcus', ' g__Lawsonella', ' g__Corynebacterium',
       ' g__Streptococcus', ' g__Haemophilus', ' g__Micrococcus',
       ' g__Lactobacillus', ' g__Cutibacterium', ' g__Kocuria', ' g__Rothia'],
      dtype='object')

In [107]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Compute the mean for each matching row and column
merged_tbl = (V1V3_tbl_matched + V4_tbl_matched) / 2

# Sort columns by total mean (descending order)
merged_tbl = merged_tbl[merged_tbl.mean().sort_values(ascending=False).index]

# Display merged DataFrame
merged_tbl


,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,1181.5,205.5,225.0,353.0,111.0,68.5,281.0,8.5,8.0,43.0
LAMI.RD001.D0.C2,905.5,253.5,430.5,371.0,168.5,63.5,269.0,6.0,173.5,49.5
LAMI.RD001.D14.C1,1117.0,268.5,703.5,243.5,354.5,61.5,88.0,2.5,88.5,13.5
LAMI.RD001.D14.C2,880.5,290.5,501.0,451.5,178.5,35.0,257.0,5.5,24.5,39.0
LAMI.RD001.D28.C1,1184.0,349.0,528.5,321.0,337.0,47.0,76.0,5.0,50.0,24.0
LAMI.RD001.D28.C2,1146.0,526.5,630.5,224.0,456.5,48.5,62.5,7.0,20.5,22.5
LAMI.RD007.D14.C1,1823.5,1047.0,148.5,91.5,183.5,20.0,39.0,0.0,3.5,15.5
LAMI.RD007.D14.C2,1799.0,783.5,189.5,84.0,253.5,31.5,70.5,0.0,0.0,0.0
LAMI.RD004.D0.C1,1340.0,352.5,664.5,290.0,35.0,52.0,8.5,2.5,16.0,29.5
LAMI.RD006.D14.C2,1700.0,387.0,458.5,65.5,208.0,78.0,7.5,16.5,66.0,1.5


In [108]:
# Convert to relative abundance (row-wise normalization)
merged_tbl_relative = merged_tbl.div(merged_tbl.sum(axis=1), axis=0)
merged_tbl_relative

,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,0.475453,0.082696,0.090543,0.142052,0.044668,0.027565,0.113078,0.003421,0.003219,0.017304
LAMI.RD001.D0.C2,0.336555,0.094220,0.160007,0.137893,0.062628,0.023602,0.099981,0.002230,0.064486,0.018398
LAMI.RD001.D14.C1,0.379803,0.091295,0.239204,0.082795,0.120537,0.020911,0.029922,0.000850,0.030092,0.004590
LAMI.RD001.D14.C2,0.330642,0.109087,0.188134,0.169546,0.067030,0.013143,0.096508,0.002065,0.009200,0.014645
LAMI.RD001.D28.C1,0.405271,0.119459,0.180900,0.109875,0.115352,0.016088,0.026014,0.001711,0.017114,0.008215
LAMI.RD001.D28.C2,0.364446,0.167435,0.200509,0.071235,0.145174,0.015424,0.019876,0.002226,0.006519,0.007155
LAMI.RD007.D14.C1,0.540777,0.310498,0.044039,0.027135,0.054419,0.005931,0.011566,0.000000,0.001038,0.004597
LAMI.RD007.D14.C2,0.560174,0.243967,0.059007,0.026156,0.078935,0.009809,0.021952,0.000000,0.000000,0.000000
LAMI.RD004.D0.C1,0.480201,0.126321,0.238129,0.103924,0.012543,0.018635,0.003046,0.000896,0.005734,0.010572
LAMI.RD006.D14.C2,0.568847,0.129496,0.153421,0.021917,0.069600,0.026100,0.002510,0.005521,0.022085,0.000502


In [109]:
merged_tbl_relative.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Corynebacterium',
       ' g__Streptococcus', ' g__Lawsonella', ' g__Micrococcus',
       ' g__Haemophilus', ' g__Kocuria', ' g__Lactobacillus', ' g__Rothia'],
      dtype='object')

In [110]:
# Save as biom for mmvec
merged_tbl_relative_transposed = merged_tbl_relative.T
output_path = f'../Data/multi-omics/16S_MERGED-Avg-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(merged_tbl_relative_transposed, output_path)

In [111]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_MERGED_tbl = metaB_tbl[metaB_tbl.index.isin(merged_tbl_relative.index)]

# Remove the 'group' column (only for VIP table)
# metaB_MERGED_tbl = metaB_MERGED_tbl.drop(columns=['group'])
metaB_MERGED_tbl.index.name = None

In [112]:
# Sort columns by total sum (descending order)
metaB_MERGED_tbl = metaB_MERGED_tbl[metaB_MERGED_tbl.sum().sort_values(ascending=False).index]

metaB_MERGED_tbl = metaB_MERGED_tbl.drop(columns=['Nicotine', 'Paracetamol'])
metaB_MERGED_tbl

,DL-Panthenol,Pyroglutamic acid,Tyrosine,(Iso)leucine,Phenylalanine,Tryptophan,C19H36O4 Fatty alcohol,Cyclo(his-pro),NCGC00380271-01,Uridine,N-Oleoylethanolamine,Urocanic acid,Glutamyltyrosine,Sorbitane Monooleate,Phenylbenzimidazole sulfonic acid,xylometazoline,Nobiletin,3-O-methylated flavonoids,Sinensetin,Gln-C14:0
LAMI.RD001.D0.C1,1473068.10,2987383.80,1942604.50,1.043807e+06,970487.50,567510.06,611709.700,302670.840,125692.3300,50026.5230,16778.8380,34717.20300,12406.82600,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D14.C1,504255.75,3269865.80,2762642.00,1.692151e+06,1496841.80,706262.44,1140515.800,651436.250,0.0000,58904.5040,5087.4270,41376.66300,43799.37000,0.000,17267.4320,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D14.C2,3137497.20,2047720.40,2397076.20,1.683097e+06,1258087.60,717058.00,263219.340,218272.830,31278.8980,48919.3750,44030.5740,5812.41000,9307.76500,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D0.C2,4475920.50,4255321.50,3177169.00,1.571455e+06,1358777.60,723508.56,944232.100,568705.940,76490.3800,75755.8400,7174.1070,24103.65150,22472.37500,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D28.C1,374392.06,5109647.00,3612457.50,1.765775e+06,1802260.60,1300446.40,269075.660,835043.000,42245.0160,113404.5400,21749.9180,45317.27550,88350.94000,36139.850,2983.9258,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D28.C2,242698.45,3283120.20,2124453.20,1.060078e+06,927238.75,508464.70,424329.250,431731.720,18491.5880,36083.3360,18348.3830,50925.83600,31103.10500,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD006.D14.C1,17990890.00,2010665.90,1216507.20,4.549094e+05,335306.12,195284.60,7236.388,81698.730,12493.9800,26096.1300,44704.6600,3965.73925,0.00000,0.000,36492.6170,0.000,0.0000,0.0,0.0,0.0
LAMI.RD006.D14.C2,18368216.00,1987542.20,1573511.80,6.725093e+05,547077.70,332766.40,15614.556,123767.820,6616.6740,31627.9860,67289.3100,19056.80800,4269.93550,11288.382,59558.3240,0.000,0.0000,0.0,0.0,0.0
LAMI.RD004.D28.C2,192005.80,481245.75,550417.06,1.089889e+05,97621.54,66863.65,88451.336,130940.270,0.0000,4855.5938,0.0000,0.00000,0.00000,20156.625,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD007.D14.C1,592174.94,1282058.00,1872823.50,5.549585e+05,472200.03,172225.81,339311.470,74992.780,107611.1200,35118.5270,18898.0270,4200.68950,6149.22800,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0


In [75]:
# Convert to relative abundance (row-wise normalization)
metaB_MERGED_tbl_relative = metaB_MERGED_tbl.div(metaB_MERGED_tbl.sum(axis=1), axis=0)
metaB_MERGED_tbl_relative

,DL-Panthenol,Pyroglutamic acid,Tyrosine,(Iso)leucine,Phenylalanine,Tryptophan,C19H36O4 Fatty alcohol,Cyclo(his-pro),NCGC00380271-01,Uridine,N-Oleoylethanolamine,Urocanic acid,Glutamyltyrosine,Sorbitane Monooleate,Phenylbenzimidazole sulfonic acid,xylometazoline,Nobiletin,3-O-methylated flavonoids,Sinensetin,Gln-C14:0
LAMI.RD001.D0.C1,0.145289,0.294647,0.191600,0.102951,0.095720,0.055974,0.060333,0.029853,0.012397,0.004934,0.001655,0.003424,0.001224,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD001.D14.C1,0.040697,0.263903,0.222966,0.136569,0.120807,0.057001,0.092048,0.052576,0.000000,0.004754,0.000411,0.003339,0.003535,0.000000,0.001394,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD001.D14.C2,0.264514,0.172638,0.202091,0.141897,0.106066,0.060453,0.022191,0.018402,0.002637,0.004124,0.003712,0.000490,0.000785,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD001.D0.C2,0.259007,0.246242,0.183852,0.090935,0.078628,0.041867,0.054640,0.032909,0.004426,0.004384,0.000415,0.001395,0.001300,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD001.D28.C1,0.024281,0.331380,0.234282,0.114517,0.116884,0.084339,0.017451,0.054156,0.002740,0.007355,0.001411,0.002939,0.005730,0.002344,0.000194,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD001.D28.C2,0.026504,0.358534,0.232002,0.115766,0.101259,0.055527,0.046339,0.047147,0.002019,0.003940,0.002004,0.005561,0.003397,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD006.D14.C1,0.802582,0.089697,0.054269,0.020294,0.014958,0.008712,0.000323,0.003645,0.000557,0.001164,0.001994,0.000177,0.000000,0.000000,0.001628,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD006.D14.C2,0.771103,0.083438,0.066056,0.028232,0.022966,0.013970,0.000656,0.005196,0.000278,0.001328,0.002825,0.000800,0.000179,0.000474,0.002500,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD004.D28.C2,0.110250,0.276332,0.316051,0.062582,0.056055,0.038393,0.050789,0.075186,0.000000,0.002788,0.000000,0.000000,0.000000,0.011574,0.000000,0.000000,0.000000,0.0,0.0,0.0
LAMI.RD007.D14.C1,0.107031,0.231723,0.338499,0.100305,0.085347,0.031129,0.061328,0.013554,0.019450,0.006347,0.003416,0.000759,0.001111,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0


In [76]:
# Save as biom for mmvec
metaB_MERGED_tbl_relative_transposed = metaB_MERGED_tbl_relative.T
output_path = f'../Data/multi-omics/metaB_MERGED-Avg-tbl_matched_relative_{skin_group}.biom'
save_as_biom(metaB_MERGED_tbl_relative_transposed, output_path)